# Digital PDF — Multi Pass Extraction

Extract structured data from a **digital** (text-based) PDF using multi-pass extraction.

Multi-pass sends each page to the LLM separately, then merges results. This is useful for long documents where context might be lost in a single call.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
import sys

from pydantic import BaseModel

from doc_intelligence import PDFExtractionMode, PDFProcessor
from doc_intelligence.pdf.types import ParseStrategy

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from utils import show_pdf_with_bboxes

PDF_URL = "https://example-files.online-convert.com/document/pdf/example.pdf"
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output")
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
class License(BaseModel):
    license_name: str

In [4]:
processor = PDFProcessor(
    provider="openai",
    model="gpt-5-mini",
    include_citations=True,
    extraction_mode=PDFExtractionMode.MULTI_PASS,
    strategy=ParseStrategy.DIGITAL,
)

In [5]:
response = processor.extract(PDF_URL, License)

2026-04-06 23:50:34.010 | INFO     | doc_intelligence.pdf.processor:extract:71 - Document parsed successfully
2026-04-06 23:50:34.011 | DEBUG    | doc_intelligence.llm:generate:41 - OpenAILLM: generate: model=gpt-5-mini
2026-04-06 23:50:36.953 | DEBUG    | doc_intelligence.pdf.extractor:_run_multi_pass:157 - PDFExtractor: multi-pass: pass1 complete: license_name='Attribution-ShareAlike 3.0 Unported'
2026-04-06 23:50:36.954 | DEBUG    | doc_intelligence.llm:generate:41 - OpenAILLM: generate: model=gpt-5-mini
2026-04-06 23:50:39.853 | DEBUG    | doc_intelligence.pdf.extractor:_run_multi_pass:166 - PDFExtractor: multi-pass: pass2 page_map: {'license_name': [0]}
2026-04-06 23:50:39.854 | INFO     | doc_intelligence.pdf.formatter:format_document_for_llm:102 - Formatting 1 pages
2026-04-06 23:50:39.855 | DEBUG    | doc_intelligence.llm:generate:41 - OpenAILLM: generate: model=gpt-5-mini
2026-04-06 23:50:44.258 | DEBUG    | doc_intelligence.pdf.extractor:_run_multi_pass:177 - PDFExtractor: mu

In [6]:
response

ExtractionResult(data=License(license_name='Attribution-ShareAlike 3.0 Unported'), metadata={'license_name': {'value': 'Attribution-ShareAlike 3.0 Unported', 'citations': [{'page': 0, 'bboxes': [{'x0': 0.20106913928643427, 'top': 0.8587326111744586, 'x1': 0.5648947389639185, 'bottom': 0.8718454960091222}]}]}})

In [7]:
assert response.metadata is not None
show_pdf_with_bboxes(
    PDF_URL,
    response.metadata["license_name"]["citations"][0],
    out_file=os.path.join(OUT_DIR, "digital_multi_pass_bbox.png"),
)

  -> saved to /Users/zeel/Public/ms/open_source/document_ai/notebooks/pdf/output/digital_multi_pass_bbox.png
